In [1]:
import numpy as np
from scipy.sparse import random, diags, csr_matrix
import Solve_Lanczos as lanczos
np.random.seed(42)

In [2]:
def generate_random_matrix(n, density=0.01):
    mat = random(n, n, density=density, format="csr", dtype=np.float64)
    row_sums = np.array(np.abs(mat).sum(axis=1)).flatten()
    diagonal_values = row_sums + 1.0
    diag_mat = diags(diagonal_values, 0)
    invertible_mat = mat + diag_mat
    return invertible_mat

In [3]:
N = 2000
test = generate_random_matrix(N, density=0.04)
test = test.dot(test.T)

In [4]:
test_mat = test.toarray()
assert np.allclose(test_mat, test_mat.T, atol=1e-8), "Matrix is not symmetric"
assert np.all(np.linalg.eigvals(test_mat) > 0), "Matrix is not positive definite"

In [5]:
x0 = np.ones(N)
max_iter = 500
num_cores = 1
num_eigenvals = 2
find_max = False
result = lanczos.solve_lanczos(test.data, test.indices, test.indptr, x0, max_iter, num_eigenvals, find_max, num_cores)

In [6]:
eigvals, eigvecs = np.linalg.eigh(test_mat)

In [7]:
def sign_invariant_error(v, w):
    return min(np.linalg.norm(v - w), np.linalg.norm(v + w))

for i in range(num_eigenvals):
    idx = -(i + 1) if find_max else i
    larger_smaller = "Largest" if find_max else "Smallest"

    computed_eval = result[i][0]
    target_eval = eigvals[idx]
    computed_evec = np.asarray(result[i][1])
    target_evec = eigvecs[:, idx]

    eigenval_error = abs(computed_eval - target_eval)
    eigenvec_error = np.linalg.norm(computed_evec - target_evec)
    eigenvec_error_sign_corrected = sign_invariant_error(computed_evec, target_evec)

    print(f"Result for {i + 1} {larger_smaller} Eigenpairs")
    print(f"Eigenvalue: Computed = {computed_eval}, Actual = {target_eval}, Error = {eigenval_error}")
    print(f"Eigenvector Error = {eigenvec_error}")

if num_eigenvals > 1:
    eigval_gap = abs(result[0][0] - result[1][0])
    print(f"Returned-eigenvalue gap (pair 1 vs 2): {eigval_gap}")

Result for 1 Smallest Eigenpairs
Eigenvalue: Computed = 645.3903076070137, Actual = 645.3903076070145, Error = 7.958078640513122e-13
Eigenvector Error = 4.1387487180829763e-13
Result for 2 Smallest Eigenpairs
Eigenvalue: Computed = 647.9250251259786, Actual = 647.9250251259751, Error = 3.410605131648481e-12
Eigenvector Error = 4.281830986678954e-13
Returned-eigenvalue gap (pair 1 vs 2): 2.5347175189648397


In [8]:
x0 = np.ones(N)
max_iter = 500
num_cores = 2
num_eigenvals = 2
find_max = True
result = lanczos.solve_lanczos(test.data, test.indices, test.indptr, x0, max_iter, num_eigenvals, find_max, num_cores)

In [9]:
for i in range(num_eigenvals):
    idx = -(i + 1) if find_max else i
    larger_smaller = "Largest" if find_max else "Smallest"

    computed_eval = result[i][0]
    target_eval = eigvals[idx]
    computed_evec = np.asarray(result[i][1])
    target_evec = eigvecs[:, idx]

    eigenval_error = abs(computed_eval - target_eval)
    eigenvec_error = np.linalg.norm(computed_evec - target_evec)
    eigenvec_error_sign_corrected = sign_invariant_error(computed_evec, target_evec)

    print(f"Result for {i + 1} {larger_smaller} Eigenpairs")
    print(f"Eigenvalue: Computed = {computed_eval}, Actual = {target_eval}, Error = {eigenval_error}")
    print(f"Eigenvector Error = {eigenvec_error}")

if num_eigenvals > 1:
    eigval_gap = abs(result[0][0] - result[1][0])
    print(f"Returned-eigenvalue gap (pair 1 vs 2): {eigval_gap}")

Result for 1 Largest Eigenpairs
Eigenvalue: Computed = 6848.324135636958, Actual = 6848.324135636957, Error = 9.094947017729282e-13
Eigenvector Error = 1.0239057449261284e-15
Result for 2 Largest Eigenpairs
Eigenvalue: Computed = 3573.531970864392, Actual = 3573.531970864388, Error = 4.092726157978177e-12
Eigenvector Error = 2.8724408583744863e-15
Returned-eigenvalue gap (pair 1 vs 2): 3274.792164772566
